# 01 — استكشاف البيانات (EDA): تصنيف الأسئلة الطبية العربية
**الشخص 1** · البيانات: أسئلة من altibbi.com · الهدف: نفهم البيانات ونكتشف مشاكلها قبل التنظيف والنموذج.

**فهرس الدفتر**
1. التجهيز وتحميل البيانات
2. الأساسيات: الحجم والأعمدة وعيّنة عشوائية
3. توزيع الفئات + تدقيق أسماء الفئات
4. أطوال الأسئلة
5. الكلمات الأكثر تكراراً
6. الأنماط النصية (regex)
7. التكرار والتشابه
8. التسريب (leakage) بين train / validation / test
9. الخلاصة وما نسلّمه للشخص 2 والشخص 3

كل قسم: **المشكلة ← الطريقة ← قبل/بعد ← ملاحظتي**.

## 1) التجهيز وتحميل البيانات

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# add the project root to the import path (the notebook lives in notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

try:                                   # for Arabic plot labels
    import arabic_reshaper, bidi       # noqa: F401
except ImportError:
    !python -m pip install -q arabic-reshaper python-bidi

from common import Columns, DATA_DIR, RANDOM_STATE, load_data
from eda_tools import (ArabicPlotter, ArabicText, ClassDistribution, DuplicateFinder, LabelAuditor,
                       LeakageDetector, LengthProfile, PatternCounter, SimilarityFinder, SplitOverview,
                       WordFrequency)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
plotter = ArabicPlotter()
print("DATA_DIR =", DATA_DIR)

In [ ]:
train, val, test, label_map = load_data()
splits = {"train": train, "val": val, "test": test}
label_map

**✍️ ملاحظة عن مصدر البيانات (تكتبها بنفسك):** 
- 


## 2) الأساسيات: الحجم والأعمدة وعيّنة عشوائية

**المشكلة:** قبل أي تحليل لازم نتأكد إن الملفات كاملة وسليمة.
**الطريقة:** جدول ملخّص لكل ملف (صفوف، أعمدة، فراغات، ids فريدة) + فحص تقاطع الـ ids بين الملفات + عيّنة 30 سؤال نقراها بعيننا.

In [ ]:
overview = SplitOverview(splits)
overview.summary()

In [ ]:
overview.id_overlap()

In [ ]:
train.dtypes.to_frame('dtype').T

**عيّنة عشوائية من 30 سؤال (train)**، نقراها بعيننا قبل أي تحليل:

In [ ]:
train.sample(30, random_state=RANDOM_STATE)[[Columns.ID, Columns.TEXT, Columns.LABEL]]

**✍️ ملاحظتي (تكتبها بنفسك من الأرقام اللي طلعت فوق):**

- شو لاحظت بالأحجام، الفراغات، الـ ids، وعيّنة الأسئلة؟
- 
- 


## 3) توزيع الفئات + تدقيق أسماء الفئات

**المشكلة:** عدد الفئات الخام بالملفات 14، وملف الترميز فيه 10 فئات. شو سبب الفرق؟
**الطريقة:** نجدول كل صيغة كتابة للفئة (`LabelAuditor`)، نقارنها مع ملف الترميز مرة بالضبط ومرة بعد توحيد الهمزات، وبعدين نقارن توزيع train و validation.

In [ ]:
auditor = LabelAuditor(train, val, label_map)
audit = auditor.audit()
audit[["train", "val", "in_mapping_exactly", "key_in_mapping", "has_edge_spaces"]]

In [ ]:
print("Raw categories in train:", train[Columns.LABEL].nunique())
print("Categories after unifying hamza variants:", auditor.canonical_label_count())
print("Categories in the label mapping file:", label_map[Columns.LABEL].nunique())
print("Categories in train but missing in validation:", auditor.labels_missing_in_val())
auditor.variant_groups()

**قبل/بعد 1: الترميز بالمطابقة الحرفية مع ملف الترميز مقابل بعد توحيد الهمزات**


In [ ]:
exact = label_map.set_index(Columns.LABEL)[Columns.LABEL_ID]
key_map = {ArabicText.normalize(k): v for k, v in exact.items()}

unmapped = train[Columns.LABEL].map(exact).isna()
print(f"Exact match: {unmapped.sum():,} rows ({unmapped.mean():.1%}) have no id in the label mapping file")

after = train[Columns.LABEL].map(ArabicText.normalize).map(key_map)
print(f"After unification (hamza differences removed): {after.isna().sum()} rows without an id")

### توزيع الفئات: train مقابل validation (بعد دمج الصيغ المتشابهة)

In [ ]:
raw_dist = ClassDistribution.compare(train, val, by_key=False)
merged_dist = ClassDistribution.compare(train, val, by_key=True)
merged_dist

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plotter.barh_counts(train[Columns.LABEL].value_counts(), "Before: raw categories (14)", ax=axes[0])
plotter.barh_counts(train[Columns.LABEL].map(ArabicText.normalize).value_counts(), "After: variants merged (10)", ax=axes[1])
plt.tight_layout(); plt.show()

plotter.grouped_barh(merged_dist, ["percent_train", "percent_val"], "Class share: train vs validation")
plt.show()

In [ ]:
raw_counts = train[Columns.LABEL].value_counts()
merged_counts = train[Columns.LABEL].map(ArabicText.normalize).value_counts()

print(f"Before merge: {len(raw_counts)} classes, largest/smallest = {ClassDistribution.imbalance_ratio(raw_counts):.1f}, equal share = {ClassDistribution.uniform_share(len(raw_counts))}%")
print(f"After merge : {len(merged_counts)} classes, largest/smallest = {ClassDistribution.imbalance_ratio(merged_counts):.1f}, equal share = {ClassDistribution.uniform_share(len(merged_counts))}%")
print("Largest gap between train and validation class share (percentage points):", merged_dist['abs_diff_pct'].max())

**✍️ ملاحظتي (تكتبها بنفسك من الأرقام اللي طلعت فوق):**

- ليش 14 فئة خام و10 بملف الترميز؟ شو أثر الأسماء المختلفة؟ هل الفئات متوازنة وعلى أي أساس؟ شو بتعني الأرقام للمقاييس (accuracy / Macro-F1)؟
- 
- 


## 4) أطوال الأسئلة

**المشكلة:** النماذج بتتأثر بطول النص، وفي نصوص قصيرة جداً (غير مفيدة) ونصوص طويلة جداً (ممكن تكون ضجيج).
**الطريقة:** عدد الكلمات والأحرف (توزيع عام + لكل فئة) وأقصر وأطول 10 أسئلة.

In [ ]:
lengths = LengthProfile()
train_len = lengths.add(train)
lengths.describe(train)

In [ ]:
plotter.histogram(train_len["n_words"], "Distribution of words per question (train)", "words", clip=100)
plt.show()
key = train[Columns.LABEL].map(ArabicText.normalize)
plotter.boxplot_by_class(train_len, "n_words", key, "Words per question by class", clip=100)
plt.show()
lengths.by_class(train, key)

In [ ]:
shortest, longest = lengths.extremes(train, 10)
display(shortest)
display(longest[[Columns.ID, "n_words", Columns.LABEL]])
print("Questions with fewer than 4 words:", int((train_len["n_words"] < 4).sum()))
print("Questions with more than 100 words:", int((train_len["n_words"] > 100).sum()))

**✍️ ملاحظتي (تكتبها بنفسك من الأرقام اللي طلعت فوق):**

- شو لاحظت بالأطوال (المتوسط، الذيل الطويل، الأسئلة القصيرة جداً، الفرق بين الفئات)؟ وشو أثر طريقة عدّ الكلمات؟
- 
- 


## 5) الكلمات الأكثر تكراراً

**المشكلة:** الكلمات الأكتر تكراراً غالباً كلمات وظيفية أو تحيات، وممكن تغطي على الكلمات الطبية المفيدة.
**الطريقة:** أكتر الكلمات تكراراً بشكل عام ولكل فئة، ومعرفة الكلمات المشتركة بين كل الفئات تقريباً (`WordFrequency`).

In [ ]:
words = WordFrequency()                      # after normalization
top = words.top_words(train, 25)
top

In [ ]:
plotter.barh_values(top["word"], top["count"], "Top 25 most frequent words (train)", "count")
plt.show()

In [ ]:
words.top_words_by_class(train, key, n=8)

In [ ]:
generic = words.shared_top_words(train, key, n=30, min_classes=8)
print(f"Words in the top 30 of at least 8 classes: {len(generic)}")
generic

**قبل/بعد 2: حجم المفردات قبل وبعد التطبيع**


In [ ]:
raw_vocab = WordFrequency(normalize=False).vocabulary_size(train)
norm_vocab = words.vocabulary_size(train)
print(f"Vocabulary size before (split on spaces): {raw_vocab:,}")
print(f"Vocabulary size after (normalization + word splitting): {norm_vocab:,}")
print(f"Reduction: {1 - norm_vocab / raw_vocab:.1%}")
display(WordFrequency(normalize=False).top_words(train, 12))

**✍️ ملاحظتي (تكتبها بنفسك من الأرقام اللي طلعت فوق):**

- شو لاحظت بالكلمات الأكثر تكراراً عامةً وبكل فئة؟ شو القرار المقترح بخصوص stop words ولماذا؟
- 
- 


## 6) الأنماط النصية (regex)

**المشكلة:** نصوص المرضى مكتوبة بحرية: أرقام، أحرف لاتينية، تشكيل، تطويل، تحيات، عامية…
**الطريقة:** نعدّ كم سؤال فيه كل نمط (`PatternCounter`) مع مثال.

In [ ]:
patterns = PatternCounter()
pattern_table = patterns.count(train)
pattern_table

In [ ]:
plotter.barh_values(pattern_table.index, pattern_table["percent"], "Share of questions containing each pattern (%)", "% of train questions")
plt.show()

**قبل/بعد 3: الأسئلة اللي فيها همزة على الألف قبل وبعد التوحيد**


In [ ]:
text = train[Columns.TEXT]
with_hamza = text.str.contains("[أإآ]").mean()
normalized = text.map(ArabicText.normalize)
print(f"Questions with a hamza on alef before unification: {with_hamza:.1%}")
print(f"After unification: {normalized.str.contains('[أإآ]').mean():.1%}")
print(f"Vocabulary size before/after: {WordFrequency(normalize=False).vocabulary_size(train):,} -> {words.vocabulary_size(train):,}")

**✍️ ملاحظتي (تكتبها بنفسك من الأرقام اللي طلعت فوق):**

- لكل نمط: النسبة، وهل بنحذفه أو بنوحّده أو بنتركه، ولماذا؟
- 
- 


## 7) التكرار والتشابه

**المشكلة:** الأسئلة المكررة أو المتشابهة جداً بتخلي النموذج يحفظ بدل ما يتعلم، وبتنفخ نتيجة التقييم لو وصلت لـ validation.
**الطريقة:** (أ) تكرار مطابق بالنص الخام وبعد التطبيع، داخل كل ملف؛ (ب) تضارب الفئات بين النسخ المكررة؛ (ج) تشابه جزئي بـ TF-IDF + NearestNeighbors على دفعات (الـ vectorizer يتعلم على train فقط).

In [ ]:
finder = DuplicateFinder()
rows = []
for name, df in splits.items():
    rows.append({"split": name,
                 "rows in dup groups (raw)": len(finder.within(df, normalized=False)),
                 "rows in dup groups (normalized)": len(finder.within(df, normalized=True))})
pd.DataFrame(rows).set_index("split")

In [ ]:
dups = finder.within(train, normalized=True)
dups[[Columns.ID, Columns.TEXT, Columns.LABEL]].head(12)

**تضارب الفئات:** نفس السؤال بفئتين مختلفتين

In [ ]:
conflicts = finder.label_conflicts(train)
print("Conflicting rows:", len(conflicts))
conflicts[[Columns.ID, Columns.TEXT, Columns.LABEL]]

**التشابه الجزئي** (cosine على TF-IDF؛ نقارن كل سؤال بأقرب سؤال بـ train):

In [ ]:
sim = SimilarityFinder(train)                       # fit on train only
nearest_train_self = sim.nearest_within_reference()   # train against itself
nearest_val = sim.nearest_in_reference(val)
nearest_test = sim.nearest_in_reference(test)

summary = pd.concat({
    "train (vs train)": SimilarityFinder.threshold_table(nearest_train_self["similarity"]),
    "validation (vs train)": SimilarityFinder.threshold_table(nearest_val["similarity"]),
    "test (vs train)": SimilarityFinder.threshold_table(nearest_test["similarity"]),
}, axis=1)
summary

In [ ]:
near_dups = nearest_train_self[nearest_train_self["similarity"] >= 0.90]
same = (near_dups["label"].map(ArabicText.normalize) == near_dups["neighbour_label"].map(ArabicText.normalize))
print(f"Near-duplicate pairs >= 0.90 inside train: {len(near_dups)}, same category: {same.mean():.1%}")
near_dups.sort_values("similarity", ascending=False)[["id", "neighbour_id", "similarity", "label", "neighbour_label", "text"]].head(8)

**✍️ ملاحظتي (تكتبها بنفسك من الأرقام اللي طلعت فوق):**

- شو لاحظت بالتكرار الحرفي وبعد التطبيع، تضارب الفئات، والتشابه الجزئي؟
- 
- 


## 8) التسريب (leakage) بين train / validation / test

**المشكلة:** لو سؤال موجود بـ train وبـ validation، نتيجة التقييم بتطلع أحسن من الحقيقة (النموذج حافظه).
**الطريقة:** نعدّ الأسئلة المشتركة بين كل زوج من الملفات، حرفياً وبعد التطبيع (`LeakageDetector`)، وبعدين نشوف التسريب شبه المطابق من القسم السابق.

In [ ]:
leak = LeakageDetector(train, val, test)
leak.exact_report()

In [ ]:
leaked_val = leak.val_rows_matching_train()
print("Validation rows that also appear in train:", len(leaked_val))
print("Same category:", int(leaked_val["same_label"].sum()), "| different category:", int((~leaked_val["same_label"]).sum()))
leaked_val[[Columns.ID, Columns.TEXT, Columns.LABEL, "train_labels", "same_label"]]

In [ ]:
leaked_test = leak.test_rows_matching_train()
print("Test rows that also appear in train:", len(leaked_test))
leaked_test[[Columns.ID, Columns.TEXT]]

In [ ]:
near_val = nearest_val[nearest_val["similarity"] >= 0.90]
near_test = nearest_test[nearest_test["similarity"] >= 0.90]
print(f"Similarity >= 0.90 with train -> validation: {len(near_val)} ({len(near_val)/len(val):.2%}) | test: {len(near_test)} ({len(near_test)/len(test):.2%})")
agree = (near_val["query_label"].map(ArabicText.normalize) == near_val["match_label"].map(ArabicText.normalize))
print(f"Of the similar validation questions, share with the same category as the nearest train neighbour: {agree.mean():.1%}")

**قبل/بعد 5: validation قبل وبعد حذف الصفوف المسرّبة من train**


In [ ]:
val_clean = leak.val_without_leakage()
print(f"validation before: {len(val):,} rows -> after removing rows leaked from train: {len(val_clean):,} rows (removed {len(val) - len(val_clean)})")
print(f"test: {len(test):,} rows before and after (no test row is ever removed)")

**✍️ ملاحظتي (تكتبها بنفسك من الأرقام اللي طلعت فوق):**

- القرار الخاص بالتسريب (validation / test) وسببه، مع الأرقام.
- 
- 


## 9) الخلاصة وما نسلّمه للشخص 2 والشخص 3

**✍️ تكتبها بنفسك:** جدول بالمشاكل اللي اكتشفتها (المشكلة | الرقم | القرار/الجهة) وشو بتسلّمه للشخص 2 والشخص 3.

| # | المشكلة | الرقم | القرار / الجهة |
|---|---|---|---|
|   |   |   |   |
